# Day5 - Demo CDC: Change Data Capture (Bronze → Silver)

**Objetivo:** Aplicar eventos CDC (Insert/Update/Delete) da Bronze `pedidos_cdc_raw` sobre a Silver `pedidos_cdc`, mantendo o estado final de cada pedido via `MERGE INTO`.

| Item | Valor |
|---|---|
| **Camada** | Silver (processamento de CDC) |
| **Entrada** | `{catalog}.bronze.pedidos_cdc_raw` |
| **Saída** | `{catalog}.silver.pedidos_cdc` |
| **Ordem no pipeline** | Após `DIA4_Gerador_Ingestao_Dados` (Bronze CDC) |

In [0]:
dbutils.widgets.text("catalog", "capgemini_academy", "Nome do catálogo no Unity Catalog")
catalog = dbutils.widgets.get("catalog")

## Criação da tabela Silver de destino

Define o esquema da tabela Silver `pedidos_cdc` que guardará o estado final de cada pedido (não o histórico de eventos).

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`silver`.`pedidos_cdc` (
    pedido_id STRING,
    cliente_id STRING,
    produto_id STRING,
    quantidade BIGINT,
    valor_total DOUBLE,
    data_pedido TIMESTAMP
);

## Temp view: último evento por pedido

Usa `ROW_NUMBER` para obter apenas o evento mais recente de cada `pedido_id` a partir da Bronze `pedidos_cdc_raw`. Essa view será a fonte do `MERGE`.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v_cdc_latest AS
    SELECT *
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY pedido_id ORDER BY op_ts DESC) AS rn
        FROM `${catalog}`.`bronze`.`pedidos_cdc_raw`
    )
    WHERE rn = 1;

## MERGE INTO — Aplicação do CDC

Compara a Silver de destino com a view `v_cdc_latest`. Decide linha por linha: deletar (op='D'), atualizar (op='U') ou inserir (op='I'/'U'). `MERGE` é idempotente — reexecutar com os mesmos eventos não duplica nem corrompe dados.

In [0]:
%sql
MERGE INTO `${catalog}`.`silver`.`pedidos_cdc` AS tgt
    USING v_cdc_latest AS src
        ON tgt.pedido_id = src.pedido_id
    WHEN MATCHED AND src.op = 'D'
        THEN DELETE
    WHEN MATCHED AND src.op = 'U'
        THEN UPDATE SET
            tgt.cliente_id = src.cliente_id,
            tgt.produto_id = src.produto_id,
            tgt.quantidade = src.quantidade,
            tgt.valor_total = src.valor_total,
            tgt.data_pedido = src.data_pedido
    WHEN NOT MATCHED AND src.op IN ('I','U')
        THEN INSERT (pedido_id, cliente_id, produto_id, quantidade, valor_total, data_pedido)
        VALUES (src.pedido_id, src.cliente_id, src.produto_id, src.quantidade, src.valor_total, src.data_pedido);